AI Assistance: OpenAI ChatGPT and Anthropic's Claude were used for code debugging, code generation, code organization, and code methodological brainstorming\. All final modeling, implementation, validation, commentary, and interpretation were performed and verified by the authors\.

# Make new predictions (Lasso regression)

Loads a saved Lasso regression pipeline and generates predictions on new input data at
`NEW_DATA_PATH`.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

platform = 'deepnote'

if platform == 'deepnote':
    PROCESSED_DIR = Path('data/Processed')
    MODELS_DIR = Path('models')

if platform == 'vscode':
    PROCESSED_DIR = Path('work/Processed')
    MODELS_DIR = Path('work/models')

## I. Configuration

`HORIZON` selects which saved model to use. `NEW_DATA_PATH` points at the single input file to run inference on (it must already be feature-engineered via `05a_realtime_create_time_lagged_features.ipynb`).

In [2]:
# Set the horizon length (e.g., 0-hr, 3-hr, 6-hr) and data path

HORIZON = '0hr'
NEW_DATA_PATH = PROCESSED_DIR / 'RT_linear_regression_features.parquet'  # update this with the path to the new data

HORIZON_CONFIG = {
    '0hr': {
        'model_path': MODELS_DIR / 'best_lasso_model_0hr.pkl',
        'target': 'kp_index',
    },
    '3hr': {
        'model_path': MODELS_DIR / 'best_lasso_model_3hr.pkl',
        'target': 'kp_index_3_hr_forecast',
    },
    '6hr': {
        'model_path': MODELS_DIR / 'best_lasso_model_6hr.pkl',
        'target': 'kp_index_6_hr_forecast',
    },
}

config = HORIZON_CONFIG[HORIZON]
print(f'Horizon: {HORIZON}')
print(f'Model path: {config["model_path"]}')
print(f'New data path: {NEW_DATA_PATH}')
print(f'Target column: {config["target"]}')

Horizon: 0hr
Model path: models/best_lasso_model_0hr.pkl
New data path: data/Processed/RT_linear_regression_features.parquet
Target column: kp_index


## II. Load model

In [3]:
def load_model(model_path):
    """Load a saved sklearn Pipeline (StandardScaler + Lasso) from a .pkl file."""

    model_path = Path(model_path)
    pipeline = joblib.load(model_path)

    step_names = [name for name, _ in pipeline.steps]
    lasso_alpha = pipeline.named_steps['lasso'].alpha

    print(f'Model loaded from {model_path}.')
    print(f'Pipeline steps: {step_names}')
    print(f'Lasso alpha: {lasso_alpha}')
    print(f'Expected feature count: {len(pipeline.feature_names_in_)}')

    return pipeline


In [4]:
# Load the model
model = load_model(config['model_path'])

Model loaded from models/best_lasso_model_0hr.pkl.
Pipeline steps: ['scaler', 'lasso']
Lasso alpha: 0.0008286427728546842
Expected feature count: 76
/root/venv/lib/python3.11/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/root/venv/lib/python3.11/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator Lasso from version 1.9.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/root/venv/lib/python3.11/site-packages/sklearn/base.py:348: InconsistentV

## III. Load new input data

In [5]:
def load_new_data(new_data_path):
    """Load new input data to run inference on."""

    new_data_path = Path(new_data_path)
    df_new = pd.read_parquet(new_data_path)

    print(f'New data loaded from {new_data_path}.')
    print(f'Shape: {df_new.shape}')

    return df_new

In [6]:
# Load new input data
df_new = load_new_data(NEW_DATA_PATH)

New data loaded from data/Processed/RT_linear_regression_features.parquet.
Shape: (1, 82)


## IV. Align feature columns between model and new data

Checks that all of the model's feature columns are in the new data. If any columns are missing, then an error is raised.

In [7]:
def prepare_inference_features(df_new, expected_features):
    """Align new data to the exact feature columns/order the pipeline was trained on."""

    expected_features = list(expected_features)
    missing = [col for col in expected_features if col not in df_new.columns]
    if missing:
        raise ValueError(f'New data is missing expected feature columns: {missing}')

    ignored = [col for col in df_new.columns if col not in expected_features]

    X_new = df_new[expected_features]

    rows_before = len(X_new)
    X_new = X_new.dropna()
    rows_after = len(X_new)

    print(f'Aligned to {len(expected_features)} expected feature columns.')
    print(f'Columns in new data not used as features in the model (ignored): {ignored}')
    print(f'Rows before dropping nulls: {rows_before}.')
    print(f'Rows after dropping nulls: {rows_after}.')

    return X_new

In [8]:
# Align the proper features and drop any extraneous ones not used by the model.
# Nulls are also dropped.
X_new = prepare_inference_features(df_new, model.feature_names_in_)
X_new.shape

Aligned to 76 expected feature columns.
Columns in new data not used as features in the model (ignored): ['year', 'day', 'hour', 'minute', 'minute_cos', 'minute_sin']
Rows before dropping nulls: 1.
Rows after dropping nulls: 1.


(1, 76)

## V. Make the predictions

In [9]:
def predict_kp(pipeline, X_new):
    """Generate Kp predictions for new input data, aligned to X_new's index."""

    y_pred = pd.Series(pipeline.predict(X_new), index=X_new.index, name='kp_pred')
    y_pred = y_pred.clip(lower=0)

    print(f'Generated {len(y_pred)} predictions.')
    print(f'Prediction min/mean/max: {y_pred.min():.4f} / {y_pred.mean():.4f} / {y_pred.max():.4f}')

    return y_pred

In [10]:
# Run the prediction function
y_pred = predict_kp(model, X_new)
y_pred.head()

Generated 1 predictions.
Prediction min/mean/max: 1.4545 / 1.4545 / 1.4545


datetime
2026-08-10 03:00:00    1.454528
Name: kp_pred, dtype: float64

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>